# 06 — 3-Class Fit Classifier

**Project:** TalentMatch.ai — Final Model  
**Goal:** Train a 3-class classifier that predicts resume-JD fit as:
- 🔴 **No Fit** — clearly mismatched
- 🟡 **Potential Fit** — some overlap, worth considering
- 🟢 **Good Fit** — strong alignment

## Why we pivoted from regression to classification
Continuous score prediction (0-100) requires near-perfect label consistency across all samples. Public datasets either have too few samples (Phase 2, 680 samples) or labels derived from cosine similarity rather than human judgment (Phase 4, which performed worse despite 10x more data). Classification is a more tractable problem — the model only needs to distinguish broad fit categories, not predict exact scores. This is also more useful in practice: recruiters need fit/no-fit decisions, not arbitrary numeric scores.

## Data strategy — combine two datasets
- `0xnbk/resume-ats-score-v1-en`: 6,374 samples with `original_label` (No Fit / Potential Fit / Good Fit)
- `netsol/resume-score-details`: 851 valid samples — we convert scores to classes using thresholds:
  - score < 34 → No Fit
  - score 34-66 → Potential Fit
  - score > 66 → Good Fit
- Combined: ~7,200 labeled samples

## Notebook Flow
1. Install + authenticate
2. Load and parse `0xnbk` dataset
3. Load and convert `netsol` dataset
4. Combine datasets and analyze class balance
5. Build PyTorch Dataset
6. Load DistilBERT with 3-class head
7. Train with Trainer API
8. Evaluate: accuracy, per-class F1
9. Push to HuggingFace Hub
10. Inference sanity check


In [ ]:
# =============================================================
# Step 1 — Install dependencies and authenticate
# =============================================================
# REMINDER:
#   - Set HF_TOKEN in Colab Secrets (key icon, left sidebar)
#   - Runtime must be T4 GPU
# =============================================================

!pip install -q -U transformers accelerate huggingface_hub evaluate scikit-learn

from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))
print("Authenticated with HuggingFace Hub.")

import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

In [ ]:
# =============================================================
# Step 2 — Load and parse 0xnbk dataset
# =============================================================
# This dataset already has original_label: No Fit / Potential Fit / Good Fit
# Text format: resume + ' SEP ' + jd (split on first occurrence)
# We map string labels to integers:
#   0 = No Fit
#   1 = Potential Fit
#   2 = Good Fit
# =============================================================

from datasets import load_dataset
import pandas as pd
import numpy as np
import json

LABEL_MAP = {'No Fit': 0, 'Potential Fit': 1, 'Good Fit': 2}
LABEL_NAMES = ['No Fit', 'Potential Fit', 'Good Fit']

def parse_ats_sample(sample):
    """Parse 0xnbk dataset sample. Returns None if malformed."""
    text = sample['text']
    if ' SEP ' not in text:
        return None
    parts = text.split(' SEP ', 1)
    resume = parts[0].strip()
    jd     = parts[1].strip()
    if not resume or not jd:
        return None
    label_str = sample['original_label']
    if label_str not in LABEL_MAP:
        return None
    return {
        'resume': resume,
        'jd':     jd,
        'label':  LABEL_MAP[label_str],
        'source': 'ats',
    }

dataset_ats = load_dataset("0xnbk/resume-ats-score-v1-en")

ats_records = []
for split in ['train', 'validation']:
    for s in dataset_ats[split]:
        parsed = parse_ats_sample(s)
        if parsed:
            ats_records.append(parsed)

ats_df = pd.DataFrame(ats_records)
print(f"ATS dataset: {len(ats_df)} samples")
print(f"Label distribution:")
print(ats_df['label'].map({0:'No Fit', 1:'Potential Fit', 2:'Good Fit'}).value_counts())

In [ ]:
# =============================================================
# Step 3 — Load and convert netsol dataset
# =============================================================
# The netsol dataset has continuous scores (0-100) from GPT-4o.
# We convert to 3 classes using thresholds based on the score
# distribution we analyzed in notebook 01:
#   score < 34  → No Fit       (label 0)
#   score 34-66 → Potential Fit (label 1)
#   score > 66  → Good Fit     (label 2)
#
# We also re-run the score normalization fix from notebook 01
# (dynamic max scaling, 60/40 macro/micro weighting).
# =============================================================

from huggingface_hub import snapshot_download
import glob, json

# Load raw JSON files directly (same approach as notebook 01)
snapshot_dir = snapshot_download(
    repo_id="netsol/resume-score-details", repo_type="dataset"
)
json_files = glob.glob(f"{snapshot_dir}/**/*.json", recursive=True)
raw_samples = []
for f in json_files:
    try:
        with open(f) as fp:
            raw_samples.append(json.load(fp))
    except:
        pass
print(f"Loaded {len(raw_samples)} raw samples from netsol")

def extract_netsol(sample):
    """Extract and normalize netsol sample. Returns None if invalid."""
    try:
        output = sample.get("output", {})
        if isinstance(output, str):
            output = json.loads(output)
        inp = sample.get("input", {})
        if isinstance(inp, str):
            inp = json.loads(inp)
        scores = output.get("scores", {})
        aggregated = scores.get("aggregated_scores", {})
        resume_text = inp.get("resume", "") or ""
        jd_text     = inp.get("job_description", "") or ""
        macro_score = aggregated.get("macro_scores", None)
        micro_score = aggregated.get("micro_scores", None)
        valid = output.get("valid_resume_and_jd", False)
        if not valid or not resume_text.strip() or not jd_text.strip():
            return None
        if macro_score is None or micro_score is None:
            return None
        return {
            'resume': resume_text.strip(),
            'jd':     jd_text.strip(),
            'macro':  float(macro_score),
            'micro':  float(micro_score),
        }
    except:
        return None

netsol_records = []
for s in raw_samples:
    r = extract_netsol(s)
    if r:
        netsol_records.append(r)

netsol_df = pd.DataFrame(netsol_records)

# Dynamic max normalization (same as notebook 01 fix)
macro_max = netsol_df['macro'].max()
micro_max = netsol_df['micro'].max()
netsol_df['score'] = netsol_df.apply(
    lambda r: round(np.clip(
        ((0.6 * r['macro'] / macro_max) + (0.4 * r['micro'] / micro_max)) * 100, 0, 100
    ), 2), axis=1
)

# Convert continuous score to 3-class label
def score_to_label(score):
    if score < 34:
        return 0  # No Fit
    elif score <= 66:
        return 1  # Potential Fit
    else:
        return 2  # Good Fit

netsol_df['label']  = netsol_df['score'].apply(score_to_label)
netsol_df['source'] = 'netsol'

print(f"\nNetsol dataset: {len(netsol_df)} samples")
print("Label distribution:")
print(netsol_df['label'].map({0:'No Fit', 1:'Potential Fit', 2:'Good Fit'}).value_counts())

In [ ]:
# =============================================================
# Step 4 — Combine datasets and analyze class balance
# =============================================================
# We concatenate both datasets and do a stratified 80/20 split.
# Stratification ensures all 3 classes are represented in both
# train and val sets proportionally.
#
# If any class is heavily underrepresented we note it — it may
# cause the model to underperform on that class.
# =============================================================

from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Keep only columns we need
ats_clean    = ats_df[['resume', 'jd', 'label', 'source']]
netsol_clean = netsol_df[['resume', 'jd', 'label', 'source']]

combined_df = pd.concat([ats_clean, netsol_clean], ignore_index=True)
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Combined dataset: {len(combined_df)} samples")
print(f"\nClass distribution:")
for label_id, label_name in enumerate(LABEL_NAMES):
    count = (combined_df['label'] == label_id).sum()
    pct   = round(count / len(combined_df) * 100, 1)
    print(f"  {label_name}: {count} ({pct}%)")

# Stratified split
train_df, val_df = train_test_split(
    combined_df, test_size=0.15, random_state=42, stratify=combined_df['label']
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"\nTrain: {len(train_df)} | Val: {len(val_df)}")

# Plot class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#e74c3c', '#f39c12', '#27ae60']

for ax, df, title in [(axes[0], train_df, 'Train'), (axes[1], val_df, 'Validation')]:
    counts = [( df['label'] == i).sum() for i in range(3)]
    ax.bar(LABEL_NAMES, counts, color=colors)
    ax.set_title(f'{title} Class Distribution')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================
# Step 5 — Build PyTorch Dataset
# =============================================================
# For 3-class classification, labels are integers 0, 1, or 2.
# HuggingFace automatically uses CrossEntropyLoss with num_labels=3.
# No label normalization needed — CrossEntropyLoss handles integers.
# =============================================================

import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class FitClassifierDataset(Dataset):
    """PyTorch Dataset for 3-class resume-JD fit classification."""
    def __init__(self, df, tokenizer):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encoding = self.tokenizer(
            row['resume'],
            row['jd'],
            max_length=MAX_LENGTH,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(row['label'], dtype=torch.long)
        }

train_dataset = FitClassifierDataset(train_df, tokenizer)
val_dataset   = FitClassifierDataset(val_df,   tokenizer)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset:   {len(val_dataset)} samples")

sample = train_dataset[0]
print(f"\ninput_ids shape: {sample['input_ids'].shape}")
label_idx = sample['labels'].item()\n",
    "print(f\"label: {label_idx} ({LABEL_NAMES[label_idx]})\")"


In [ ]:
# =============================================================
# Step 6 — Load DistilBERT with 3-class classification head
# =============================================================
# num_labels=3 → CrossEntropyLoss, outputs logits for 3 classes.
# The classifier head: Linear(768 → 3)
# During inference we take softmax to get probabilities per class.
# =============================================================

from transformers import AutoModelForSequenceClassification
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={0: 'No Fit', 1: 'Potential Fit', 2: 'Good Fit'},
    label2id={'No Fit': 0, 'Potential Fit': 1, 'Good Fit': 2},
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


def compute_metrics(eval_pred):
    """
    Accuracy and per-class F1 for 3-class classification.
    We report macro F1 (average across all 3 classes equally)
    since all classes matter equally for our use case.
    """
    from sklearn.metrics import accuracy_score, f1_score, classification_report
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro')
    f1_per_class = f1_score(labels, predictions, average=None)
    return {
        'accuracy':          round(float(accuracy),           4),
        'f1_macro':          round(float(f1_macro),           4),
        'f1_no_fit':         round(float(f1_per_class[0]),    4),
        'f1_potential_fit':  round(float(f1_per_class[1]),    4),
        'f1_good_fit':       round(float(f1_per_class[2]),    4),
    }

print("Model and metrics defined.")

In [ ]:
# =============================================================
# Step 7 — Configure TrainingArguments and train
# =============================================================
# Classification converges faster than regression.
# We use 4 epochs — enough to learn the 3-class boundaries
# without overfitting on ~7,000 samples.
#
# metric_for_best_model='f1_macro': we save the checkpoint with
# the best average F1 across all 3 classes, not just accuracy.
# This prevents the model from gaming accuracy by ignoring
# minority classes.
#
# Expected training time: ~20-25 min on T4 GPU.
# =============================================================

from transformers import TrainingArguments, Trainer
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/TalentMatch-AI/checkpoints/three_class"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_steps=25,
    fp16=torch.cuda.is_available(),
    report_to='none',
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Starting 3-class fit classifier training...")
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Steps per epoch: {len(train_dataset) // training_args.per_device_train_batch_size}")
print("---")

train_result = trainer.train()

print("\nTraining complete!")
print(f"Total steps:   {train_result.global_step}")
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Training time: {train_result.metrics['train_runtime']:.1f}s")

In [ ]:
# =============================================================
# Step 8 — Evaluate with full classification report
# =============================================================

from sklearn.metrics import classification_report
import numpy as np

print("Evaluating on validation set...")
val_results = trainer.evaluate(eval_dataset=val_dataset)

print("\n===== 3-CLASS CLASSIFIER RESULTS =====")
print(f"Overall Accuracy: {val_results['eval_accuracy']:.4f}")
print(f"Macro F1:         {val_results['eval_f1_macro']:.4f}")
print(f"\nPer-class F1:")
print(f"  No Fit:        {val_results['eval_f1_no_fit']:.4f}")
print(f"  Potential Fit: {val_results['eval_f1_potential_fit']:.4f}")
print(f"  Good Fit:      {val_results['eval_f1_good_fit']:.4f}")
print("======================================")

# Detailed classification report
print("\nGenerating full classification report...")
predictions_output = trainer.predict(val_dataset)
preds = predictions_output.predictions.argmax(axis=-1)
labels = predictions_output.label_ids

print(classification_report(labels, preds, target_names=LABEL_NAMES))

if val_results['eval_accuracy'] >= 0.80:
    print("✓ Accuracy >= 80% — classifier is ready for deployment.")
else:
    print("~ Consider more epochs or reviewing class balance.")

In [ ]:
# =============================================================
# Step 9 — Push to HuggingFace Hub
# =============================================================
# We push as TalentMatch-AI-classifier — the final production model.
# The id2label and label2id configs are already embedded in the model
# from Step 6, so the Hub will display the class names correctly.
# =============================================================

HF_REPO = "LucasLisboadev/TalentMatch-AI-classifier"

print(f"Pushing model to: https://huggingface.co/{HF_REPO}")

model.save_pretrained("/content/talentmatch_classifier")
tokenizer.save_pretrained("/content/talentmatch_classifier")

model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

print(f"\nModel pushed successfully!")
print(f"View at: https://huggingface.co/{HF_REPO}")

In [ ]:
# =============================================================
# Step 10 — Inference sanity check
# =============================================================
# We test the same 3 pairs we've used throughout the project.
# Expected:
#   Strong match   → Good Fit (high confidence)
#   Moderate match → Potential Fit
#   Weak match     → No Fit (high confidence)
#
# We report both the predicted class AND the confidence
# (softmax probability) so the Gradio demo can show both.
# =============================================================

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

LABEL_NAMES  = ['No Fit', 'Potential Fit', 'Good Fit']
LABEL_EMOJIS = ['🔴', '🟡', '🟢']

def predict_fit(resume, jd, tokenizer, model):
    """Returns predicted class label, emoji, and confidence."""
    inputs = tokenizer(
        resume, jd,
        max_length=512, truncation=True, padding='max_length', return_tensors='pt'
    )
    with torch.no_grad():
        logits = model(**inputs).logits
    probs      = torch.softmax(logits, dim=-1)[0]
    pred_class = probs.argmax().item()
    confidence = round(probs[pred_class].item() * 100, 1)
    return LABEL_NAMES[pred_class], LABEL_EMOJIS[pred_class], confidence

inf_tokenizer = AutoTokenizer.from_pretrained(HF_REPO)
inf_model     = AutoModelForSequenceClassification.from_pretrained(HF_REPO)
inf_model.eval()

# Test pairs
strong_resume = """Senior Backend Engineer with 6 years of experience building scalable
REST APIs using Python and FastAPI. Proficient in PostgreSQL, Redis, and Docker.
Led a team of 4 engineers to deliver a microservices architecture serving 2M daily users.
Strong experience with CI/CD pipelines, AWS deployments, and agile development."""
strong_jd = """Backend Engineer: 4+ years Python required. FastAPI or Django,
PostgreSQL, Docker. Team leadership a strong plus."""

moderate_resume = """Marketing Manager with 5 years in B2B demand generation,
content strategy, Salesforce and HubSpot. Led campaigns generating $2M pipeline.
Strong analytical and communication skills."""
moderate_jd = """Sales Director: 5+ years in B2B sales or marketing.
Salesforce CRM, pipeline management, executive presentations required."""

weak_resume = """Registered Nurse with 8 years ICU experience. Patient care,
medication administration, Epic EHR certified. ACLS/BLS certified."""
weak_jd = """Backend Engineer: 4+ years Python required.
FastAPI or Django, PostgreSQL, Docker."""

print("===== INFERENCE SANITY CHECK =====")
for name, resume, jd in [
    ("Strong match",   strong_resume,   strong_jd),
    ("Moderate match", moderate_resume, moderate_jd),
    ("Weak match",     weak_resume,     weak_jd),
]:
    label, emoji, conf = predict_fit(resume, jd, inf_tokenizer, inf_model)
    print(f"{name}: {emoji} {label} ({conf}% confidence)")

print("\nExpected: Good Fit > Potential Fit > No Fit")

## Summary

| Metric | Value |
|---|---|
| Task | 3-class classification (No Fit / Potential Fit / Good Fit) |
| Training data | ~7,200 samples (0xnbk + netsol combined) |
| Overall Accuracy | TBD |
| Macro F1 | TBD |
| HF Hub | LucasLisboadev/TalentMatch-AI-classifier |

*(Update after running)*

**Next step:** Update `app.py` in the HuggingFace Space to use the classifier model.
